# Integrated concordance × GRCh38 Sankey
Figures A/B/C derived from pre-summed flows (medians across assemblies).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RESULTS_DIR = Path('../results')
PLUS_DIR = RESULTS_DIR / 'intermediate_spreadsheets' / 'sankey_plus_divergence'
OUTPUT_DIR = Path('figures')
OUTPUT_DIR.mkdir(exist_ok=True)

flows = pd.read_csv(PLUS_DIR / 'flows_medians.tsv', sep='\t')
print('Adjacencies:', flows['adjacency'].unique())

COLORS = {
    'RBH pass': '#2e86c1', 'No RBH': '#e74c3c',
    'Full': '#2e86c1', 'Partial': '#85c1e9', 'None': '#e74c3c',
    'protein_coding':'#1f77b4', 'lncRNA':'#9467bd', 'pseudogene':'#8c564b', 'other_ncRNA':'#2ca02c', 'other':'#7f7f7f',
    'both_agree_reference':'#2ecc71', 'both_agree_diverged':'#3498db', 'ensembl_specific_divergence':'#e67e22', 'cat_specific_divergence':'#9b59b6', 'insufficient_data':'#95a5a6',
    'Intact':'#2ecc71', 'Partial disruption':'#f1c40f', 'Disrupted':'#e74c3c'
}

def stacked_bar_with_ribbons(adj1, adj2, left_labels, right_labels, left_colors, right_colors, title, filename):
    # Build left/right proportions from medians
    L = flows[flows['adjacency']==adj1].groupby('from')['median'].sum().reindex(left_labels).fillna(0)
    R = flows[flows['adjacency']==adj2].groupby('to')['median'].sum().reindex(right_labels).fillna(0)
    Ltot, Rtot = L.sum(), R.sum()
    l_pct = (L / Ltot * 100).fillna(0)
    r_pct = (R / Rtot * 100).fillna(0)

    # Ranges
    def ranges(vals):
        out, cur = {}, 0.0
        for k, v in vals.items():
            out[k] = (cur, cur + float(v))
            cur += float(v)
        return out
    Lr = ranges(l_pct.to_dict())
    Rr = ranges(r_pct.to_dict())

    fig, ax = plt.subplots(figsize=(12, 7))
    bar_h = 0.5
    # Draw bars
    y_left, y_right = 1, 0
    left = 0.0
    for k in left_labels:
        v = l_pct[k]
        ax.barh(y_left, v, left=left, height=bar_h, color=left_colors.get(k, '#999'))
        left += v
    left = 0.0
    for k in right_labels:
        v = r_pct[k]
        ax.barh(y_right, v, left=left, height=bar_h, color=right_colors.get(k, '#bbb'))
        left += v

    # Ribbons from adjacency data connecting L→R
    from matplotlib.patches import Polygon
    sub = flows[flows['adjacency']==adj2]  # use right adjacency for detailed mapping
    cursL = {k: Lr[k][0] for k in left_labels}
    cursR = {k: Rr[k][0] for k in right_labels}
    # Normalize contributions per segment
    denL = sub.groupby('from')['median'].sum().reindex(left_labels).fillna(0).to_dict()
    denR = sub.groupby('to')['median'].sum().reindex(right_labels).fillna(0).to_dict()
    for _, row in sub.iterrows():
        a, b, w = row['from'], row['to'], float(row['median'])
        if w <= 0 or denL.get(a,0)==0 or denR.get(b,0)==0:
            continue
        wl = (w / denL[a]) * (Lr[a][1]-Lr[a][0])
        wr = (w / denR[b]) * (Rr[b][1]-Rr[b][0])
        x0, x1 = cursL[a], cursL[a]+wl
        x2, x3 = cursR[b], cursR[b]+wr
        cursL[a], cursR[b] = x1, x3
        ax.add_patch(Polygon([(x0, y_left-bar_h/2), (x1, y_left-bar_h/2), (x3, y_right+bar_h/2), (x2, y_right+bar_h/2)],
                          closed=True, facecolor=left_colors.get(a,'#999'), alpha=0.25))
    ax.set_xlim(0, 100)
    ax.set_yticks([1,0]); ax.set_yticklabels(['Left','Right'])
    ax.set_xlabel('Percentage of row denominator (%)')
    ax.set_title(title)
    plt.tight_layout(); fig.savefig(OUTPUT_DIR/filename, dpi=300, bbox_inches='tight'); plt.show()

# Figure A: RBH → Concordance → Biotype → GRCh38
left_labels = ['RBH pass','No RBH']
right_labels = ['Full','Partial','None']
left_colors = {'RBH pass': '#2e86c1', 'No RBH': '#e74c3c'}
right_colors = {'Full':'#2e86c1','Partial':'#85c1e9','None':'#e74c3c'}
stacked_bar_with_ribbons('A_RBH_to_Concordance','A_Concordance_to_Biotype', left_labels, ['protein_coding','lncRNA','pseudogene','other_ncRNA','other'], left_colors, {k:COLORS[k] for k in ['protein_coding','lncRNA','pseudogene','other_ncRNA','other']}, 'A1. RBH → Concordance → Biotype', 'figure_mainX_A1.png')
stacked_bar_with_ribbons('A_Concordance_to_Biotype','A_Biotype_to_GRCh38', ['Full','Partial','None'], ['both_agree_reference','both_agree_diverged','ensembl_specific_divergence','cat_specific_divergence','insufficient_data'], {k:COLORS[k] for k in ['Full','Partial','None']}, {k:COLORS[k] for k in ['both_agree_reference','both_agree_diverged','ensembl_specific_divergence','cat_specific_divergence','insufficient_data']}, 'A2. Concordance → Biotype → GRCh38', 'figure_mainX_A2.png')

# Figure B: + CDS (protein_coding only)

FileNotFoundError: [Errno 2] No such file or directory: '../results/intermediate_spreadsheets/sankey_plus_divergence/flows_medians.tsv'

In [ ]:

# Figure B: + CDS (protein_coding only)
# Build pseudo flows: Biotype→CDS and CDS→GRCh38 from B_* adjacencies

def stacked_simple(adj, left_labels, right_labels, title, filename):
    L = flows[flows['adjacency']==adj].groupby('from')['median'].sum().reindex(left_labels).fillna(0)
    R = flows[flows['adjacency']==adj].groupby('to')['median'].sum().reindex(right_labels).fillna(0)
    Ltot, Rtot = L.sum(), R.sum()
    l_pct = (L/Ltot*100).fillna(0); r_pct = (R/Rtot*100).fillna(0)
    def ranges(vals):
        out, cur = {}, 0.0
        for k, v in vals.items(): out[k]=(cur,cur+float(v)); cur+=float(v)
        return out
    Lr, Rr = ranges(l_pct.to_dict()), ranges(r_pct.to_dict())
    fig, ax = plt.subplots(figsize=(12,6)); bar_h=0.5
    yL,yR=1,0; left=0
    for k in left_labels:
        v=l_pct[k]; ax.barh(yL,v,left=left,height=bar_h,color=COLORS.get(k,'#777')); left+=v
    left=0
    for k in right_labels:
        v=r_pct[k]; ax.barh(yR,v,left=left,height=bar_h,color=COLORS.get(k,'#777')); left+=v
    from matplotlib.patches import Polygon
    sub=flows[flows['adjacency']==adj]
    cursL={k:Lr[k][0] for k in left_labels}; cursR={k:Rr[k][0] for k in right_labels}
    denL=sub.groupby('from')['median'].sum().reindex(left_labels).fillna(0).to_dict()
    denR=sub.groupby('to')['median'].sum().reindex(right_labels).fillna(0).to_dict()
    for _,row in sub.iterrows():
        a,b,w=row['from'],row['to'],float(row['median'])
        if w<=0 or denL.get(a,0)==0 or denR.get(b,0)==0: continue
        wl=(w/denL[a])*(Lr[a][1]-Lr[a][0]); wr=(w/denR[b])*(Rr[b][1]-Rr[b][0])
        x0,x1=cursL[a],cursL[a]+wl; x2,x3=cursR[b],cursR[b]+wr; cursL[a],cursR[b]=x1,x3
        ax.add_patch(Polygon([(x0,yL-bar_h/2),(x1,yL-bar_h/2),(x3,yR+bar_h/2),(x2,yR+bar_h/2)],closed=True,facecolor=COLORS.get(a,'#999'),alpha=0.25))
    ax.set_xlim(0,100); ax.set_yticks([1,0]); ax.set_yticklabels(['Left','Right']); ax.set_xlabel('% of row denominator'); ax.set_title(title)
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/filename,dpi=300,bbox_inches='tight'); plt.show()

stacked_simple('B_Biotype_to_CDS', ['protein_coding'], ['intact','partial','disrupted'], 'B1. Biotype (protein_coding) → CDS integrity', 'figure_mainX_B1.png')
stacked_simple('B_CDS_to_GRCh38', ['intact','partial','disrupted'], ['both_agree_reference','both_agree_diverged','ensembl_specific_divergence','cat_specific_divergence','insufficient_data'], 'B2. CDS integrity → GRCh38 (protein_coding)', 'figure_mainX_B2.png')

# Figure C: Concordance → GRCh38 (overall)
stacked_bar_with_ribbons('A_Concordance_to_Biotype','C_Concordance_to_GRCh38', ['Full','Partial','None'], ['both_agree_reference','both_agree_diverged','ensembl_specific_divergence','cat_specific_divergence','insufficient_data'], {k:COLORS[k] for k in ['Full','Partial','None']}, {k:COLORS[k] for k in ['both_agree_reference','both_agree_diverged','ensembl_specific_divergence','cat_specific_divergence','insufficient_data']}, 'C. Concordance → GRCh38 (overall)', 'figure_mainX_C_overall.png')
